[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_notebooks/blob/main/session4a_student.ipynb)

# Session 5 — Dimensionality Reduction: PCA, t-SNE and UMAP

High-dimensional data often have far fewer real degrees of freedom than they have features. **Dimensionality reduction** looks for a low-dimensional description: for compression, for visualisation, or as input for another method.

Each method makes a choice about **what to preserve**, and that choice decides what you can and cannot read from its output. To see this clearly we use **one simulated data set** throughout, whose true structure we know exactly.

Each **✏️ Exercise** starts with a prediction. Write it down *before* you run the code.

In the next notebook we apply the same methods to real SDSS spectra.

In [ ]:
import warnings
for msg in ["n_jobs value", "IProgress not found", "The default value of `copy`"]:
    warnings.filterwarnings("ignore", message=msg)

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import HDBSCAN
import umap

plt.rcParams.update({
    "figure.figsize": (7, 5), "figure.dpi": 110, "font.size": 12, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 2, "legend.frameon": False,
})

## The data

We start from three hidden coordinates and place three structures in them:

* a **sequence**: 1200 points along a smooth, curved arc (a stretched helix). Each point has a position $s \in [0, 1]$ along the arc;
* **blob A**: 300 points in a compact Gaussian group near one end of the arc;
* **blob B**: 300 points in a Gaussian group about **twice as wide** as A (after noise) and about **three times farther** from the arc.

We then embed these 3D points into **10 dimensions** with a random rotation, and add Gaussian noise with $\sigma = 0.3$ to **every** one of the 10 features. The result, `X`, is what the methods see: a $1800 \times 10$ table. The labels `part` (0 = sequence, 1 = A, 2 = B) and the position `s` are only used to colour the plots.

In [ ]:
rng = np.random.default_rng(7)
NOISE = 0.3

s = rng.uniform(0, 1, 1200)
sequence = np.c_[4*np.cos(2.5*np.pi*s), 4*np.sin(2.5*np.pi*s), 6*s] + rng.normal(0, 0.05, (1200, 3))
blob_A = np.array([0, 0, 9]) + rng.normal(0, 0.4, (300, 3))
blob_B = np.array([0, 0, -12]) + rng.normal(0, 1.2, (300, 3))
hidden = np.vstack([sequence, blob_A, blob_B])
part = np.repeat([0, 1, 2], [1200, 300, 300])

rotation, _ = np.linalg.qr(rng.normal(size=(10, 3)))        # orthonormal map from 3 to 10 dimensions
X = hidden @ rotation.T + rng.normal(0, NOISE, (len(hidden), 10))
print("X:", X.shape)

def plot_map(ax, Z, title=""):
    # sequence coloured by its position s; blob A orange; blob B black
    ax.scatter(*Z[part == 0].T, c=s, cmap="viridis", s=4, linewidth=0)
    ax.scatter(*Z[part == 1].T, color="C1", s=4, linewidth=0, label="blob A")
    ax.scatter(*Z[part == 2].T, color="black", s=4, linewidth=0, label="blob B")
    ax.set(title=title, xticks=[], yticks=[])

fig = plt.figure(figsize=(16, 4.4))
ax = fig.add_subplot(1, 4, 1, projection="3d")
ax.scatter(*hidden[part == 0].T, c=s, cmap="viridis", s=3)
ax.scatter(*hidden[part == 1].T, color="C1", s=3)
ax.scatter(*hidden[part == 2].T, color="black", s=3)
ax.set_title("the hidden 3D structure")
ax.view_init(15, -60)
for i, (a, b) in enumerate([(0, 1), (2, 3), (4, 5)]):
    plot_map(fig.add_subplot(1, 4, i + 2), X[:, [a, b]], f"what we see: features {a} and {b}")
fig.axes[1].legend(markerscale=4, loc="upper left")
fig.tight_layout();

## 1. PCA

Principal component analysis finds orthogonal directions ordered by the variance of the data along them. For the data matrix with the mean of each feature subtracted, $X_c$, the **singular value decomposition** is

$$X_c = U\,S\,V^T .$$

The rows of $V^T$ are the principal components, and the variance along component $j$ is $\lambda_j = s_j^2/(N-1)$ — the eigenvalues of the covariance matrix. Keeping the first $k$ components gives the best rank-$k$ approximation of the data in the least-squares sense.

### ✏️ Exercise 1 — PCA by hand: how many dimensions does the data have?

**Predict first:** `X` has 10 features. How many eigenvalues should be clearly large? What value should the others have? (Hint: what did we add to *every* feature?)

1. Subtract the column means and compute the SVD with `np.linalg.svd(Xc, full_matrices=False)`.
2. Compute the eigenvalues `lam` and check them against `PCA().fit(X).explained_variance_`.

In [ ]:
# YOUR CODE HERE


assert np.allclose(lam, PCA().fit(X).explained_variance_)
print("eigenvalues:", np.round(lam, 3))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.semilogy(np.arange(1, 11), lam, "o-", color="C0", ms=7)
ax.axhline(NOISE**2, color="C1", ls="--", lw=1.5, label=r"noise variance $\sigma^2$")
ax.set(xlabel="component", ylabel="eigenvalue (variance)", xticks=range(1, 11), title="The eigenvalue spectrum")
ax.legend();

### ✏️ Exercise 2 — Units matter

Suppose feature 0 had been recorded in different units, so all its values are 100 times larger.

**Predict first:** where will the first principal component point, and what fraction of the variance will it explain?

1. Make `X_units`, a copy of `X` with column 0 multiplied by 100, and fit PCA.
2. Fit PCA again after `StandardScaler`.
3. The cell prints the first component and the explained variance ratios for the original, rescaled and standardised data.

When would you **not** want to standardise? (Think of data in which every feature has the same units, and in which some features are pure noise.)

In [ ]:
# YOUR CODE HERE


np.set_printoptions(suppress=True)
for name, p in [("original", PCA().fit(X)), ("feature 0 x 100", pca_units), ("standardised", pca_std)]:
    print(f"{name:16s} PC1 = {np.round(p.components_[0], 2)}\n{'':16s} explained variance ratio = {np.round(p.explained_variance_ratio_[:4], 3)}")

**Take-away.** PCA finds the directions of largest variance *in the units you give it*. Standardising gives every feature equal weight — right when the units are arbitrary, wrong when the differences in variance are meaningful (for example, a flux spectrum in which some pixels are mostly noise).

## 2. Non-linear maps: t-SNE and UMAP

**t-SNE** (van der Maaten & Hinton 2008) converts the distances from each point to its neighbours into probabilities, and places the points in 2D so that the neighbour probabilities there match. Its main parameter, the **perplexity**, is roughly the number of neighbours each point takes into account.

**UMAP** (McInnes et al. 2018) builds a graph that connects each point to its `n_neighbors` nearest neighbours and finds a 2D layout that preserves the graph. It is faster than t-SNE and can place new points into an existing map with `.transform()`.

Both are designed to preserve **local neighbourhoods**. PCA, projected onto its first two components, preserves the directions of largest **global** variance.

In [ ]:
maps = {
    "PCA": PCA(n_components=2).fit_transform(X),
    "t-SNE": TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X),
    "UMAP": umap.UMAP(n_neighbors=15, random_state=0).fit_transform(X),
}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (name, Z) in zip(axes, maps.items()):
    plot_map(ax, Z, name)
axes[0].legend(markerscale=4, loc="lower left")
fig.tight_layout();

### ✏️ Exercise 3 — Local vs. global

Two scores tell us what each map keeps:

* **Neighbour preservation** (local): for each point, the fraction of its $k = 10$ nearest neighbours in the 10D data that are also among its 10 nearest neighbours in the map.
* **Distance correlation** (global): the rank correlation between all pairwise distances in the 10D data and in the map. Large distances should stay large.

**Predict first:** which method wins on each score? Look at the maps: where does PCA put points on top of each other that are not neighbours in 10D?

Complete `neighbour_preservation(X, Z, k)`. *Hint:* `NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X, return_distance=False)` returns each point's own index in the first column, followed by its $k$ neighbours.

In [ ]:
def neighbour_preservation(X, Z, k=10):
    # YOUR CODE HERE
    raise NotImplementedError


def distance_correlation(X, Z, n=600, seed=0):
    idx = np.random.default_rng(seed).choice(len(X), n, replace=False)
    return spearmanr(pdist(X[idx]), pdist(Z[idx]))[0]

print(f"{'':6s} {'neighbours kept':>16s} {'distance correlation':>22s}")
for name, Z in maps.items():
    print(f"{name:6s} {neighbour_preservation(X, Z):16.2f} {distance_correlation(X, Z):22.2f}")

### ✏️ Exercise 4 — Do sizes and distances in the map mean anything?

We built blob B to be about twice as wide as blob A, and about three times farther from the sequence.

**Predict first:** in each map, how much bigger will B look than A, and how much farther from the sequence?

Complete `geometry(Z)` so that it returns two ratios:
1. **size**: the RMS distance of B's points from B's centre, divided by the same quantity for A;
2. **distance**: the distance from B's centre to the nearest sequence point, divided by the same for A.

The cell applies it to the 10D data and to the three maps.

In [ ]:
def geometry(Z):
    # YOUR CODE HERE
    raise NotImplementedError


print(f"{'':10s} {'size B / A':>11s} {'distance B / A':>15s}")
for name, Z in {"10D data": X, **maps}.items():
    r_size, r_dist = geometry(Z)
    print(f"{name:10s} {r_size:11.1f} {r_dist:15.1f}")

### ✏️ Exercise 5 — Clusters that do not exist

A tempting workflow is to make a map and then find groups in it with a clustering algorithm. Our data contain exactly **three** structures, and the sequence is **one continuous** structure with no gaps.

**Predict first:** how many clusters will HDBSCAN (from the clustering session) find in a UMAP map made with very few neighbours?

1. For `n_neighbors` in 3, 5, 15 and 50, make a UMAP map of `X` (`random_state=0`), run `HDBSCAN(min_cluster_size=20)` on the **map**, and store the maps and labels in the dictionaries `umaps` and `labels`.
2. The cell counts the clusters in total and how many pieces the sequence has been cut into.
3. Repeat with `random_state=1`. Do the "clusters" stay the same?

In [ ]:
n_neighbors_list = [3, 5, 15, 50]
umaps, labels = {}, {}

# YOUR CODE HERE


fig, axes = plt.subplots(1, len(n_neighbors_list), figsize=(18, 4.4))
for ax, nn in zip(axes, n_neighbors_list):
    lab = labels[nn]
    n_total = lab.max() + 1
    n_seq = len(set(lab[(part == 0) & (lab >= 0)]))
    ax.scatter(*umaps[nn][lab < 0].T, s=3, color="#d8d8d3", linewidth=0)
    ax.scatter(*umaps[nn][lab >= 0].T, s=4, c=lab[lab >= 0], cmap="tab20", linewidth=0)
    ax.set(title=f"n_neighbors = {nn}: {n_total} clusters\n(sequence cut into {n_seq})", xticks=[], yticks=[])
fig.tight_layout();

## Summary

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| linear? | yes | no | no |
| preserves | global variance, large distances | local neighbourhoods | local neighbourhoods, a little global structure |
| sizes and distances in the map | meaningful | not meaningful | not meaningful |
| deterministic | yes | no (seed) | no (seed) |
| back to the original space | yes (`inverse_transform`) | no | approximately |
| new data | yes (`transform`) | no | yes (`transform`) |
| main knob | number of components | perplexity | `n_neighbors`, `min_dist` |

**Rules of thumb**
1. Start with PCA. The eigenvalue spectrum tells you how many dimensions carry structure and where the noise begins.
2. Decide consciously whether to standardise.
3. Use t-SNE/UMAP to *look* at data — often on the first few PCA components, which removes the noise dimensions first. Read neighbourhoods from the map, not sizes or distances.
4. Vary perplexity / `n_neighbors` and the seed. Trust only structure that survives, and confirm it in the original space.